# Robustness + tail analysis of the late-fusion hybrid

1. **Robustness** — repeat (tune beta on val, report on held-out test) over 5 user splits; report mean +/- std of the lift, so +12.6% isn't one lucky split.
2. **Tail analysis** — where does audio help? Split each user's relevant items by train popularity and compare fused vs SASRec on head vs tail.

One SASRec is trained and reused for both. Accelerator → **GPU**, Internet On.

In [ ]:
!pip install -q --no-cache-dir --upgrade "git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
import os
os.environ["YMREC_DATA"] = "/kaggle/working/data"
import time, numpy as np, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("CUDA:", torch.cuda.is_available())

from ymrec.data.sequences import build_sequences
from ymrec.data.embeddings import load_aligned_embeddings
data = build_sequences(size="50m", maxlen=200)
content_emb, content_mask, dim = load_aligned_embeddings(data.item_ids)
print(f"items={data.n_items:,} eval={len(data.eval_pos):,} coverage={content_mask.mean():.3f}")

In [ ]:
from ymrec.models.sasrec import train_and_eval
t0 = time.time()
sasrec, _ = train_and_eval(data, d=64, n_blocks=2, n_heads=1, dropout=0.2,
                           epochs=120, batch_size=128, lr=1e-3, eval_every=60)
print(f"SASRec trained in {(time.time()-t0)/60:.1f} min")

In [ ]:
# 1) Robustness over 5 user splits.
from ymrec.models.fusion import robustness
rob = robustness(sasrec, content_emb, data, seeds=(0, 1, 2, 3, 4))
s = rob["summary"]
print(f"\nSASRec  NDCG@10 = {s['sasrec_mean']:.4f} +/- {s['sasrec_std']:.4f}")
print(f"Fused   NDCG@10 = {s['fused_mean']:.4f} +/- {s['fused_std']:.4f}")
print(f"Lift            = {s['lift_mean_%']:+.1f}% +/- {s['lift_std_%']:.1f}%   (betas: {s['betas']})")

In [ ]:
# 2) Where does content help? (tail = few train interactions)
from ymrec.models.fusion import tail_analysis
import pandas as pd
rows = tail_analysis(sasrec, content_emb, data, beta=0.75, thresholds=(5, 20, 100))
pd.DataFrame(rows)